
# Football Detection Benchmark + Stable ID Notebook

Bu notebook hazır modellerle video üzerinde:
- Oyuncu / top tespiti
- ByteTrack tracking
- Stable ID düzeltme
- Oyuncu altında yuvarlak marker
- Video çıktısı
- CSV tracking log
- Model benchmark özeti

üretir.

> Önce `VIDEO_PATH`, `OUTPUT_DIR`, model ayarları ve opsiyonel Roboflow API bilgilerini düzenle.


In [1]:

# ============================================================
# CELL 1 - Kurulum
# Pillow dosyalarının Colab'da eski/yeni sürüm karışmasını önlemek için
# ilgili paketler temiz biçimde ve uyumlu sürümlerle kurulur.
# İlk çalıştırmada Colab oturumu bir kez otomatik yeniden başlar.
# ============================================================
import os
import sys
import subprocess
from pathlib import Path

_restart_marker = Path("/tmp/football_detect_dependencies_ready")

if not _restart_marker.exists():
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "--upgrade", "--force-reinstall", "--no-cache-dir",
        "requests==2.32.4",
        "Pillow==11.3.0",
        "opencv-python-headless>=4.8,<4.12",
        "pandas>=2.0",
        "tqdm>=4.66",
        "scikit-learn>=1.3",
        "ultralytics>=8.3,<9",
        "supervision>=0.25,<0.28",
    ])
    _restart_marker.touch()
    print("Paketler kuruldu. Temiz import için oturum yeniden başlatılıyor...")
    os.kill(os.getpid(), 9)
else:
    print("Paketler hazır. Sonraki hücreye geçebilirsin.")


Paketler hazır. Sonraki hücreye geçebilirsin.


In [2]:

# ============================================================
# CELL 2 - Ayarlar
# ============================================================
import os
from pathlib import Path

# Video yolunu buraya yaz
# Colab örnek: /content/input_video.mp4
# Kaggle örnek: /kaggle/input/xxx/video.mp4
VIDEO_PATH = "/content/icardi.mp4"

OUTPUT_DIR = "/content/football_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Test edilecek modeller
# YOLO modelleri otomatik indirilir.
# Daha hızlı için yolo11n.pt / yolo11s.pt, daha iyi doğruluk için yolo11x.pt kullan.
MODELS_TO_TEST = [
    {"name": "yolo11x_coco", "type": "ultralytics", "weights": "yolo11x.pt"},
    {"name": "yolov8x_coco", "type": "ultralytics", "weights": "yolov8x.pt"},
    # {"name": "yolov10x_coco", "type": "ultralytics", "weights": "yolov10x.pt"},

    # Roboflow kullanmak istersen aşağıyı aç:
    # {"name": "roboflow_football_v11", "type": "roboflow", "model_id": "football-players-detection-3zvbc/11"},
]

# Roboflow opsiyonel. Kullanmayacaksan boş bırak.
ROBOFLOW_API_KEY = "3xHrGr1zui9h2t7SOVCG"  # örnek: "xxxxxxxxxxxxxxxx"
ROBOFLOW_API_URL = "https://serverless.roboflow.com"

# Detection ayarları
DETECT_EVERY_N_FRAMES = 1       # 1 en doğru, 2/3 daha hızlı ama kaçırma artabilir
YOLO_PLAYER_CONF = 0.25         # COCO person için
YOLO_BALL_CONF = 0.08           # COCO sports ball için düşük tutulabilir
ROBOFLOW_DEFAULT_CONF = 0.15
IOU_THRESHOLD = 0.50
IMG_SIZE = 1280                 # 960 hızlı, 1280 dengeli, 1536 daha doğru ama yavaş

# Tracking / Stable ID ayarları
TRACK_ACTIVATION_THRESHOLD = 0.20
LOST_TRACK_BUFFER = 60
MIN_MATCHING_THRESHOLD = 0.80
STABLE_ID_MAX_CENTER_DISTANCE = 120
STABLE_ID_MAX_MISSING_FRAMES = 90

# Görselleştirme
DRAW_PLAYER_ELLIPSE = True
DRAW_BOX = False
SHOW_TRACK_ID = True
SHOW_CONFIDENCE = False
DRAW_BALL = True
SAVE_CSV = True
PROCESS_MAX_FRAMES = None       # test için 300 yazabilirsin, tüm video için None

print("Ayarlar hazır.")


Ayarlar hazır.


In [3]:

# ============================================================
# CELL 3 - Importlar ve yardımcı fonksiyonlar
# ============================================================
import cv2
import math
import time
import json
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from collections import defaultdict, deque
from IPython.display import display

import supervision as sv
from ultralytics import YOLO

try:
    from inference_sdk import InferenceHTTPClient
except Exception:
    InferenceHTTPClient = None


def check_video(path):
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Video bulunamadı: {path}\n"
            "VIDEO_PATH değişkenini doğru video yoluyla güncelle."
        )
    cap = cv2.VideoCapture(path)
    if not cap.isOpened():
        raise RuntimeError(f"Video açılamadı: {path}")
    fps = float(cap.get(cv2.CAP_PROP_FPS))
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    if not np.isfinite(fps) or fps <= 0:
        fps = 25.0
        print("Uyarı: Geçersiz FPS bilgisi; 25 FPS kullanılacak.")
    if w <= 0 or h <= 0:
        raise RuntimeError(f"Video boyutları okunamadı: {w}x{h}")
    print(f"Video: {path}")
    print(f"FPS: {fps:.2f} | Size: {w}x{h} | Frames: {total}")
    return fps, w, h, total


def safe_int(x):
    try:
        return int(x)
    except Exception:
        return -1


def box_center_xyxy(xyxy):
    x1, y1, x2, y2 = xyxy
    return np.array([(x1 + x2) / 2.0, (y1 + y2) / 2.0], dtype=float)


def ellipse_points_from_box(xyxy):
    x1, y1, x2, y2 = map(int, xyxy)
    cx = int((x1 + x2) / 2)
    bottom_y = int(y2)
    width = max(8, int((x2 - x1) * 0.50))
    height = max(4, int(width * 0.28))
    return (cx, bottom_y), (width, height)


def draw_text_with_bg(frame, text, org, font_scale=0.55, thickness=1):
    x, y = org
    font = cv2.FONT_HERSHEY_SIMPLEX
    (tw, th), base = cv2.getTextSize(text, font, font_scale, thickness)
    cv2.rectangle(frame, (x, y - th - base - 4), (x + tw + 6, y + base + 2), (0, 0, 0), -1)
    cv2.putText(frame, text, (x + 3, y - 3), font, font_scale, (255, 255, 255), thickness, cv2.LINE_AA)

print("Importlar hazır.")


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Importlar hazır.


In [4]:

# ============================================================
# CELL 4 - Stable ID Manager
# ByteTrack ID değişse bile yakın konum + yakın zaman mantığıyla daha stabil ID verir.
# ============================================================
class StableIDManager:
    def __init__(self, max_center_distance=120, max_missing_frames=90):
        self.max_center_distance = max_center_distance
        self.max_missing_frames = max_missing_frames
        self.next_stable_id = 1
        self.tracker_to_stable = {}
        self.stable_state = {}  # stable_id -> dict(center, frame_idx, class_name, hits)
        self.updated_in_frame = set()

    def begin_frame(self):
        self.updated_in_frame = set()

    def update(self, tracker_id, xyxy, frame_idx, class_name="player"):
        center = box_center_xyxy(xyxy)

        if tracker_id in self.tracker_to_stable:
            stable_id = self.tracker_to_stable[tracker_id]
            if stable_id in self.updated_in_frame:
                stable_id = None
            elif stable_id not in self.stable_state:
                self.tracker_to_stable.pop(tracker_id, None)
                stable_id = None

        else:
            stable_id = None

        if stable_id is not None:
            self.stable_state[stable_id] = {
                "center": center,
                "frame_idx": frame_idx,
                "class_name": class_name,
                "hits": self.stable_state.get(stable_id, {}).get("hits", 0) + 1,
            }
            self.updated_in_frame.add(stable_id)
            return stable_id

        # Eski stable ID'lerden en yakın olanı bul
        best_id = None
        best_dist = float("inf")
        for sid, state in self.stable_state.items():
            if sid in self.updated_in_frame:
                continue
            if state.get("class_name") != class_name:
                continue
            age = frame_idx - state.get("frame_idx", -10**9)
            if age < 0 or age > self.max_missing_frames:
                continue
            dist = float(np.linalg.norm(center - state["center"]))
            if dist < best_dist and dist <= self.max_center_distance:
                best_dist = dist
                best_id = sid

        if best_id is None:
            best_id = self.next_stable_id
            self.next_stable_id += 1

        self.tracker_to_stable[tracker_id] = best_id
        self.stable_state[best_id] = {
            "center": center,
            "frame_idx": frame_idx,
            "class_name": class_name,
            "hits": self.stable_state.get(best_id, {}).get("hits", 0) + 1,
        }
        self.updated_in_frame.add(best_id)
        return best_id

    def cleanup(self, frame_idx):
        remove_stable = []
        for sid, state in self.stable_state.items():
            if frame_idx - state.get("frame_idx", 0) > self.max_missing_frames * 3:
                remove_stable.append(sid)
        for sid in remove_stable:
            self.stable_state.pop(sid, None)
        valid_stable_ids = set(self.stable_state)
        self.tracker_to_stable = {
            tracker_id: stable_id
            for tracker_id, stable_id in self.tracker_to_stable.items()
            if stable_id in valid_stable_ids
        }

print("StableIDManager hazır.")


StableIDManager hazır.


In [5]:

# ============================================================
# CELL 5 - Detector sınıfları
# ============================================================
class UltralyticsDetector:
    def __init__(self, weights, imgsz=1280):
        self.weights = weights
        self.imgsz = imgsz
        self.model = YOLO(weights)
        self.names = self.model.names

    def predict(self, frame):
        results = self.model.predict(
            frame,
            imgsz=self.imgsz,
            conf=min(YOLO_PLAYER_CONF, YOLO_BALL_CONF),
            iou=IOU_THRESHOLD,
            verbose=False,
            device=0 if self._has_cuda() else None,
        )[0]

        xyxy_list, conf_list, class_id_list = [], [], []
        if results.boxes is None:
            return sv.Detections.empty(), []

        boxes = results.boxes.xyxy.cpu().numpy()
        confs = results.boxes.conf.cpu().numpy()
        cls_ids = results.boxes.cls.cpu().numpy().astype(int)

        readable = []
        for box, conf, cls_id in zip(boxes, confs, cls_ids):
            if isinstance(self.names, dict):
                class_name = self.names.get(int(cls_id), str(cls_id))
            else:
                class_name = self.names[int(cls_id)] if int(cls_id) < len(self.names) else str(cls_id)

            # COCO: person=0, sports ball=32
            keep = False
            mapped_class_id = None
            mapped_class_name = None

            if class_name == "person" and conf >= YOLO_PLAYER_CONF:
                keep = True
                mapped_class_id = 0
                mapped_class_name = "player"
            elif class_name == "sports ball" and conf >= YOLO_BALL_CONF:
                keep = True
                mapped_class_id = 1
                mapped_class_name = "ball"

            if keep:
                xyxy_list.append(box)
                conf_list.append(float(conf))
                class_id_list.append(mapped_class_id)
                readable.append(mapped_class_name)

        if len(xyxy_list) == 0:
            return sv.Detections.empty(), []

        detections = sv.Detections(
            xyxy=np.array(xyxy_list, dtype=np.float32),
            confidence=np.array(conf_list, dtype=np.float32),
            class_id=np.array(class_id_list, dtype=int),
        )
        return detections, readable

    @staticmethod
    def _has_cuda():
        try:
            import torch
            return torch.cuda.is_available()
        except Exception:
            return False


class RoboflowDetector:
    def __init__(self, model_id):
        if not ROBOFLOW_API_KEY:
            raise ValueError("ROBOFLOW_API_KEY boş. Roboflow modeli kullanmak için API key gir.")
        if InferenceHTTPClient is None:
            raise ImportError("inference-sdk kurulu değil.")
        self.model_id = model_id
        self.client = InferenceHTTPClient(api_url=ROBOFLOW_API_URL, api_key=ROBOFLOW_API_KEY)

    def predict(self, frame):
        # Roboflow SDK genelde image path / numpy kabul eder. Sorun çıkarsa temp image kullanılır.
        try:
            result = self.client.infer(frame, model_id=self.model_id)
        except Exception:
            tmp_path = "/tmp/rf_frame.jpg"
            cv2.imwrite(tmp_path, frame)
            result = self.client.infer(tmp_path, model_id=self.model_id)

        xyxy_list, conf_list, class_id_list, readable = [], [], [], []
        predictions = result.get("predictions", []) if isinstance(result, dict) else []

        for p in predictions:
            conf = float(p.get("confidence", 0.0))
            if conf < ROBOFLOW_DEFAULT_CONF:
                continue
            cname = str(p.get("class", "")).lower()
            x, y, w, h = float(p["x"]), float(p["y"]), float(p["width"]), float(p["height"])
            x1, y1, x2, y2 = x - w / 2, y - h / 2, x + w / 2, y + h / 2

            if "ball" in cname:
                mapped_class_id = 1
                mapped_class_name = "ball"
            else:
                mapped_class_id = 0
                mapped_class_name = "player"

            xyxy_list.append([x1, y1, x2, y2])
            conf_list.append(conf)
            class_id_list.append(mapped_class_id)
            readable.append(mapped_class_name)

        if len(xyxy_list) == 0:
            return sv.Detections.empty(), []

        detections = sv.Detections(
            xyxy=np.array(xyxy_list, dtype=np.float32),
            confidence=np.array(conf_list, dtype=np.float32),
            class_id=np.array(class_id_list, dtype=int),
        )
        return detections, readable


def build_detector(model_cfg):
    if model_cfg["type"] == "ultralytics":
        return UltralyticsDetector(model_cfg["weights"], imgsz=IMG_SIZE)
    if model_cfg["type"] == "roboflow":
        return RoboflowDetector(model_cfg["model_id"])
    raise ValueError(f"Bilinmeyen model type: {model_cfg['type']}")

print("Detector sınıfları hazır.")


Detector sınıfları hazır.


In [6]:

# ============================================================
# CELL 6 - Video işleme fonksiyonu
# ============================================================
def create_tracker(fps):
    return sv.ByteTrack(
        track_activation_threshold=TRACK_ACTIVATION_THRESHOLD,
        lost_track_buffer=LOST_TRACK_BUFFER,
        minimum_matching_threshold=MIN_MATCHING_THRESHOLD,
        frame_rate=max(1, int(round(fps))),
    )


def annotate_frame(frame, detections, stable_ids, class_names):
    annotated = frame.copy()

    for idx in range(len(detections)):
        xyxy = detections.xyxy[idx]
        conf = detections.confidence[idx] if detections.confidence is not None else 0.0
        class_id = int(detections.class_id[idx]) if detections.class_id is not None else 0
        tracker_id = int(detections.tracker_id[idx]) if detections.tracker_id is not None else -1
        stable_id = stable_ids[idx] if idx < len(stable_ids) else tracker_id
        class_name = class_names[idx] if idx < len(class_names) else ("ball" if class_id == 1 else "player")

        x1, y1, x2, y2 = map(int, xyxy)

        if class_name == "ball":
            if DRAW_BALL:
                cx, cy = int((x1+x2)/2), int((y1+y2)/2)
                radius = max(5, int(max(x2-x1, y2-y1)/2))
                cv2.circle(annotated, (cx, cy), radius, (255, 255, 255), 2)
                cv2.circle(annotated, (cx, cy), 2, (255, 255, 255), -1)
            continue

        if DRAW_PLAYER_ELLIPSE:
            center, axes = ellipse_points_from_box(xyxy)
            cv2.ellipse(annotated, center, axes, 0, 0, 360, (255, 255, 255), 2)

        if DRAW_BOX:
            cv2.rectangle(annotated, (x1, y1), (x2, y2), (255, 255, 255), 2)

        label_parts = []
        if SHOW_TRACK_ID:
            label_parts.append(f"ID {stable_id}")
        if SHOW_CONFIDENCE:
            label_parts.append(f"{conf:.2f}")
        if label_parts:
            draw_text_with_bg(annotated, " | ".join(label_parts), (x1, max(20, y1 - 4)))

    return annotated


def process_video_with_model(model_cfg):
    model_name = model_cfg["name"]
    print("\n==============================")
    print(f"Model başlıyor: {model_name}")
    print(f"==============================")

    fps, width, height, total_frames = check_video(VIDEO_PATH)
    if PROCESS_MAX_FRAMES is not None:
        total_to_process = min(total_frames, PROCESS_MAX_FRAMES)
    else:
        total_to_process = total_frames

    detector = build_detector(model_cfg)
    tracker = create_tracker(fps)
    stable_manager = StableIDManager(
        max_center_distance=STABLE_ID_MAX_CENTER_DISTANCE,
        max_missing_frames=STABLE_ID_MAX_MISSING_FRAMES,
    )

    output_video_path = os.path.join(OUTPUT_DIR, f"{model_name}_stable_tracking.mp4")
    output_csv_path = os.path.join(OUTPUT_DIR, f"{model_name}_tracking.csv")

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))
    if not writer.isOpened():
        raise RuntimeError("VideoWriter açılamadı. OUTPUT_DIR veya codec sorunu olabilir.")

    cap = cv2.VideoCapture(VIDEO_PATH)
    rows = []
    stats = defaultdict(int)
    last_detections = sv.Detections.empty()
    last_class_names = []
    start_time = time.time()

    pbar = tqdm(total=total_to_process, desc=model_name)
    frame_idx = 0

    while frame_idx < total_to_process:
        ok, frame = cap.read()
        if not ok:
            break

        stable_manager.begin_frame()
        should_detect = (frame_idx % max(1, DETECT_EVERY_N_FRAMES) == 0)
        if should_detect:
            detections, class_names = detector.predict(frame)
            last_detections = detections
            last_class_names = class_names
        else:
            detections, class_names = last_detections, last_class_names

        # Tracking için sadece player class'ını veriyoruz. Top genelde tracker'da stabil olmaz.
        player_mask = np.array([cid == 0 for cid in detections.class_id], dtype=bool) if len(detections) else np.array([], dtype=bool)
        ball_mask = np.array([cid == 1 for cid in detections.class_id], dtype=bool) if len(detections) else np.array([], dtype=bool)

        player_dets = detections[player_mask] if len(detections) else sv.Detections.empty()
        ball_dets = detections[ball_mask] if len(detections) else sv.Detections.empty()

        tracked_players = tracker.update_with_detections(player_dets) if len(player_dets) else sv.Detections.empty()

        stable_ids = []
        tracked_class_names = []
        if len(tracked_players):
            for i in range(len(tracked_players)):
                tid = int(tracked_players.tracker_id[i]) if tracked_players.tracker_id is not None else -1
                sid = stable_manager.update(tid, tracked_players.xyxy[i], frame_idx, "player")
                stable_ids.append(sid)
                tracked_class_names.append("player")

                x1, y1, x2, y2 = tracked_players.xyxy[i]
                conf = float(tracked_players.confidence[i]) if tracked_players.confidence is not None else 0.0
                rows.append({
                    "frame": frame_idx,
                    "time_sec": frame_idx / fps if fps else 0,
                    "model": model_name,
                    "class": "player",
                    "tracker_id": tid,
                    "stable_id": sid,
                    "confidence": conf,
                    "x1": float(x1), "y1": float(y1), "x2": float(x2), "y2": float(y2),
                    "cx": float((x1+x2)/2), "cy": float((y1+y2)/2),
                })

        # Ball detection'ları annotate için player tracking ile birleştiriyoruz
        combined_dets = tracked_players
        combined_stable_ids = stable_ids[:]
        combined_class_names = tracked_class_names[:]

        if len(ball_dets):
            # ball için tracker_id olmadığı için stable id -1
            ball_dets.tracker_id = np.full(len(ball_dets), -1, dtype=int)
            if len(combined_dets):
                combined_dets = sv.Detections.merge([combined_dets, ball_dets])
            else:
                combined_dets = ball_dets
            combined_stable_ids += [-1] * len(ball_dets)
            combined_class_names += ["ball"] * len(ball_dets)

            for i in range(len(ball_dets)):
                x1, y1, x2, y2 = ball_dets.xyxy[i]
                conf = float(ball_dets.confidence[i]) if ball_dets.confidence is not None else 0.0
                rows.append({
                    "frame": frame_idx,
                    "time_sec": frame_idx / fps if fps else 0,
                    "model": model_name,
                    "class": "ball",
                    "tracker_id": -1,
                    "stable_id": -1,
                    "confidence": conf,
                    "x1": float(x1), "y1": float(y1), "x2": float(x2), "y2": float(y2),
                    "cx": float((x1+x2)/2), "cy": float((y1+y2)/2),
                })

        stats["frames"] += 1
        stats["player_detections"] += len(tracked_players)
        stats["ball_detections"] += len(ball_dets)
        stats["unique_stable_ids"] = max(stats["unique_stable_ids"], stable_manager.next_stable_id - 1)

        annotated = annotate_frame(frame, combined_dets, combined_stable_ids, combined_class_names)
        writer.write(annotated)

        stable_manager.cleanup(frame_idx)
        frame_idx += 1
        pbar.update(1)

    pbar.close()
    cap.release()
    writer.release()

    elapsed = time.time() - start_time
    out_fps = stats["frames"] / elapsed if elapsed > 0 else 0

    df = pd.DataFrame(rows)
    if SAVE_CSV:
        df.to_csv(output_csv_path, index=False)

    summary = {
        "model": model_name,
        "frames_processed": int(stats["frames"]),
        "processing_fps": round(out_fps, 2),
        "avg_players_per_frame": round(stats["player_detections"] / max(1, stats["frames"]), 2),
        "avg_ball_detections_per_frame": round(stats["ball_detections"] / max(1, stats["frames"]), 3),
        "unique_stable_player_ids": int(stats["unique_stable_ids"]),
        "output_video": output_video_path,
        "output_csv": output_csv_path,
    }

    print("\nÖzet:")
    print(json.dumps(summary, indent=2, ensure_ascii=False))
    return summary, df

print("Video işleme fonksiyonu hazır.")


Video işleme fonksiyonu hazır.


In [7]:

# ============================================================
# CELL 7 - Tüm modelleri çalıştır
# ============================================================
all_summaries = []
all_dfs = []

for model_cfg in MODELS_TO_TEST:
    try:
        summary, df = process_video_with_model(model_cfg)
        all_summaries.append(summary)
        all_dfs.append(df)
    except Exception as e:
        print(f"\nHATA - {model_cfg['name']}: {type(e).__name__}: {e}")

benchmark_df = pd.DataFrame(all_summaries)
benchmark_path = os.path.join(OUTPUT_DIR, "benchmark_summary.csv")
benchmark_df.to_csv(benchmark_path, index=False)

print("\nBenchmark tamamlandı:")
display(benchmark_df)
print("Benchmark CSV:", benchmark_path)



Model başlıyor: yolo11x_coco
Video: /content/icardi.mp4
FPS: 25.00 | Size: 1920x1080 | Frames: 625


yolo11x_coco:   0%|          | 0/625 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:

# ============================================================
# CELL 8 - Çıktıları listele
# ============================================================
print("OUTPUT_DIR:", OUTPUT_DIR)
for p in sorted(Path(OUTPUT_DIR).glob("*")):
    print(p)



## Önerilen kullanım

1. Önce hızlı test için:
   ```python
   PROCESS_MAX_FRAMES = 300
   MODELS_TO_TEST = [{"name": "yolo11x_coco", "type": "ultralytics", "weights": "yolo11x.pt"}]
   ```
2. Çıktı güzel görünüyorsa tüm video için:
   ```python
   PROCESS_MAX_FRAMES = None
   ```
3. Daha hızlı ama biraz daha az doğru test için:
   ```python
   IMG_SIZE = 960
   weights = "yolo11s.pt"
   ```
4. Daha doğru ama daha yavaş test için:
   ```python
   IMG_SIZE = 1536
   weights = "yolo11x.pt"
   ```
